# Echo-State Network (ESN) — Frozen Reservoir + Readout-Only Training

This notebook demonstrates **reservoir computing** on the neurogym *PerceptualDecisionMaking* task:

- The **input projection** (`input2h`) and the **recurrent reservoir** (`h2h`) are kept **frozen** during training.
- Only the **readout layer** (`readout_layer`) is optimized.
- The recurrent reservoir is initialized at criticality using the recipe from *Pachitariu et al. (2026)* (`critical_init`), i.e. the recurrent weight matrix is scaled so that its largest real eigenvalue is close to `0.998`.

After training the critical reservoir, we:

1. Evaluate whether a readout-only network can solve the task.
2. Run a **fixed-point analysis** on the trained ESN.
3. Compare the critical reservoir with **poorly initialized reservoirs** (`emax = 0.1, 0.5, 1, 2, 5`) to show that ESN performance strongly depends on the recurrent initialization.

> Reference: Pachitariu, M., Zhong, L., Gracias, A., Minisi, M., Lopez, C., & Stringer, C. (2026). *A critical initialization for biological neural networks*. Nature. https://doi.org/10.1038/s41586-026-10528-1


In [ ]:
%matplotlib inline
import sys
sys.path.append('../src')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments
from neuralrnn import SupervisedObjective, load_dataset
from neuralrnn.analysis import fit_pca, find_fixed_points, linearize, dominant_direction

# reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cpu'  # set to 'cuda' if available

# Checkpoint and figure directories
from pathlib import Path
MODEL_DIR = Path('./models/09')
FIG_DIR = Path('./figs/09')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


## 1. Task data: Perceptual Decision-Making

We use the same neurogym task as notebook `01_ctrnn_fixedpoints_paradigmA.ipynb`. The network must integrate left/right stimulus evidence and produce a choice at the end of the trial.


In [ ]:
ds = load_dataset('perceptual_decision_making', batch_size=16, seq_len=100, dt=100)
print('input_dim =', ds.input_dim, '| n_actions =', ds.output_dim)

# visualize a few sample trials — straight from the training dataset (no ds_viz)
from neuralrnn import visualization as viz

n_show = 3
fig, _ = viz.plot_trials(ds.sample_trials(n_show), dt=ds.dt,
                         title='Perceptual decision-making: inputs and target')
fig.savefig(FIG_DIR / 'pdm_sample_trials.png', dpi=150)
plt.show()

## 2. Critical recurrent initialization

The `critical_init_recurrent` helper replicates the initialization used in the `critical_init` reference project:

1. Draw a random matrix $A$ uniformly from $[-1, 1]$.
2. Symmetrize it and remove self-connections (zero diagonal).
3. Normalize by $\sqrt{N}\,\mathrm{std}(A)$ and divide by 2 for the symmetric case.
4. Rescale so that the largest real eigenvalue equals `emax`:

\[
A \leftarrow A \cdot \frac{\texttt{emax}}{\max_i \operatorname{Re}[\lambda_i(A)]}
\]


In [ ]:
def critical_init_recurrent(weight, emax=0.998, symmetric=True, seed=None):
    """Initialize a recurrent weight matrix using the critical_init recipe.

    Args:
        weight: target parameter, only used to infer shape (M, M).
        emax: target largest real eigenvalue.
        symmetric: whether to symmetrize the random matrix.
        seed: optional random seed.
    Returns:
        A: (M, M) critically initialized tensor.
    """
    M = weight.shape[0]
    if seed is not None:
        torch.manual_seed(seed)
    A = 2 * torch.rand((M, M)) - 1.0
    if symmetric:
        A = (A + A.T) / 2
    A -= torch.diag(torch.diag(A))          # remove self-connections
    A /= M**0.5 * A.std()
    A /= 2.0 if symmetric else 1.0
    evals = torch.linalg.eigvals(A)
    A /= torch.real(evals).max() / emax
    return A

# quick sanity check
_test = critical_init_recurrent(torch.empty(64, 64), emax=0.998, seed=0)
_test_evals = torch.linalg.eigvals(_test)
print(f'M={_test.shape[0]}, spectral radius={torch.real(_test_evals).max():.4f}')


## 3. Build the CTRNN-ESN with a frozen reservoir

We use the framework's built-in freeze flags:

- `freeze_input=True`  → `input2h` frozen
- `freeze_recurrent=True` → `h2h` frozen
- `freeze_output=False` → `readout_layer` trainable

The `Trainer` automatically ignores parameters whose `requires_grad=False`.


In [ ]:
LATENT_DIM = 256  # reservoir size

cfg_esn = AutoConfig.for_model(
    'ctrnn',
    input_dim=ds.input_dim,
    latent_dim=LATENT_DIM,
    output_dim=ds.output_dim,
    dt=100,
    tau=100,
    nonlinearity_mode='pre_activation',
    freeze_input=True,
    freeze_recurrent=True,
    freeze_output=False,
)

# fix the random seed for the trainable readout initialization
torch.manual_seed(42)
model_esn = AutoModel.from_config(cfg_esn)

# apply critical initialization to the recurrent weights
with torch.no_grad():
    model_esn.h2h.weight.copy_(critical_init_recurrent(model_esn.h2h.weight, emax=0.998, seed=0))

# verify which parameters are trainable
for n, p in model_esn.named_parameters():
    print(f'{n:30s} requires_grad={p.requires_grad}')


## 4. Train only the readout layer

Because the reservoir is frozen, the loss gradient only updates `readout_layer`.


In [ ]:
SAVE_DIR = str(MODEL_DIR / 'ctrnn_esn_emax_0.998')
os.path.exists(SAVE_DIR)

In [ ]:
objective = SupervisedObjective(task_type='classification')



# save the critical ESN
SAVE_DIR = str(MODEL_DIR / 'ctrnn_esn_emax_0.998')

if os.path.exists(SAVE_DIR):
    # load model
    model_esn = AutoModel.from_pretrained(SAVE_DIR)
else:
    os.makedirs(SAVE_DIR, exist_ok=True)
    history_esn = Trainer(
        model_esn, ds, objective,
        TrainingArguments(max_steps=1500, learning_rate=1e-3, log_every=300)
    ).train()
    model_esn.save_pretrained(SAVE_DIR)
    print(f'Model saved to {SAVE_DIR}/')


## 5. Evaluate the critical ESN and collect hidden activity

We run 500 fresh trials, use the final-time logit to choose the action, and collect hidden-state trajectories for later PCA and fixed-point analysis.


In [ ]:
def collect_activity_and_accuracy(model, dataset):
    '''Evaluate model on a trial-aligned dataset; return accuracy, trial info, and activity.'''
    model.eval()
    with torch.no_grad():
        out = model(dataset.inputs)       # batched forward over all trials
    states_all = out.states.numpy()       # (N, T, latent_dim)
    logits_all = out.outputs.numpy()      # (N, T, output_dim)
    infos, activities, predictions = [], {}, {}
    for i, cond in enumerate(dataset.conditions):
        n = cond['n_steps']               # true length of this trial (unpadded)
        pred = int(logits_all[i, n - 1].argmax()) - 1  # 1=choice0, 2=choice1 -> 0/1 ground_truth
        infos.append(dict(cond))
        activities[i] = states_all[i, :n]
        predictions[i] = pred
    df = pd.DataFrame(infos)
    df['action'] = pd.Series(predictions)
    acc = float((df['action'] == df['ground_truth']).mean())
    return acc, df, activities


# Complete trials straight from the training dataset (no new dataset needed)
ds_eval = ds.sample_trials(500)
acc_esn, df_esn, acts_esn = collect_activity_and_accuracy(model_esn, ds_eval)
print(f'Critical ESN accuracy (emax=0.998): {acc_esn:.3f}')
print('Unique coherences:', sorted(df_esn['coh'].unique()))

### 5.1 Reservoir trajectories in PCA space

Project the frozen reservoir activity onto the first two principal components, colored by signed coherence and ground-truth choice.


In [ ]:
activity_all = np.concatenate([acts_esn[i] for i in range(len(acts_esn))], axis=0)
pca = fit_pca(activity_all, n_components=2)
print('Explained variance ratio:', np.round(pca.explained_variance_ratio, 3))

colors_rgb = np.array([[27, 158, 119], [117, 112, 179]]) / 255.0  # green, purple
cohs_pos = sorted([c for c in df_esn['coh'].unique() if c > 0])
intensity = [0.4, 0.7, 1.0, 1.3][:len(cohs_pos)]

fig = plt.figure(figsize=(4, 4))
ax = fig.add_axes([0.2, 0.2, 0.6, 0.6])

for gt in [0, 1]:
    cohs_ = cohs_pos[::-1] if gt == 0 else cohs_pos
    int_ = intensity[::-1] if gt == 0 else intensity
    for i_coh, coh in enumerate(cohs_):
        idx = df_esn[(df_esn['coh'] == coh) & (df_esn['ground_truth'] == gt)].index
        if len(idx) == 0:
            continue
        avg = np.mean([acts_esn[i] for i in idx], axis=0)
        avg_pc = pca.transform(avg)
        color = colors_rgb[gt] * int_[i_coh]
        signed_coh = coh * (2 * gt - 1)
        ax.plot(avg_pc[:, 0], avg_pc[:, 1], 'o-', color=color, ms=3,
                markeredgecolor='none', lw=1, label=f'{signed_coh:+.1f}')

ax.legend(title='Signed Coh', loc='upper left', bbox_to_anchor=(1.0, 1.0),
          frameon=False, fontsize=7)
ax.set_xlabel('PC 1', fontsize=9)
ax.set_ylabel('PC 2', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_title('Critical ESN reservoir trajectories', fontsize=10)
fig.savefig(FIG_DIR / 'esn_pca_trajectories.png', dpi=150)
plt.show()


## 6. Fixed-point analysis of the critical ESN

We search for approximate fixed points under the **0-coherence stimulus** (`fixation=1, left=0.5, right=0.5`). This is the same condition used in notebook 01. Fixed points reveal the dynamical scaffold that the readout layer exploits.


In [ ]:
task_input = torch.tensor([1.0, 0.5, 0.5], dtype=torch.float32)

fps = find_fixed_points(
    model_esn, backend='numeric', task_input=task_input,
    n_candidates=64, n_iters=10000, speed_tol=0.5
)
print(f'Found {len(fps)} fixed points')

# project fixed points into PCA space
if len(fps):
    coords_fp = pca.transform(fps.coords())
else:
    coords_fp = np.empty((0, 2))

fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.15, 0.15, 0.75, 0.75])

# overlay a subset of trial trajectories
for i in range(min(100, len(acts_esn))):
    activity_pc = pca.transform(acts_esn[i])
    color = colors_rgb[0] if df_esn.loc[i, 'ground_truth'] == 0 else colors_rgb[1]
    ax.plot(activity_pc[:, 0], activity_pc[:, 1], '.', color=color, alpha=0.8, ms=2)

if len(fps):
    ax.plot(coords_fp[:, 0], coords_fp[:, 1], 'kx', ms=5, mew=0.5, label='Fixed points')
    # dominant eigenvector at the first fixed point
    lin = linearize(model_esn, fps.points[0].z, task_input=task_input)
    d = dominant_direction(lin)
    seg = pca.transform(np.array([fps.points[0].z + 2 * d,
                                   fps.points[0].z - 2 * d]))
    ax.plot(seg[:, 0], seg[:, 1], 'r-', lw=2, label='Dominant eigenvector')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_title('Fixed points in critical ESN PCA space')
fig.savefig(FIG_DIR / 'esn_fixed_points_pca.png', dpi=150)
plt.show()


## 7. How reservoir initialization affects ESN performance

To show that the reservoir initialization — not the readout capacity — is the limiting factor, we train **readout-only** CTRNNs whose recurrent weights are scaled to different `emax` values:

- `emax < 1` (e.g. `0.1`, `0.5`): the reservoir is damped and forgets inputs quickly.
- `emax = 1`: exactly at the stability boundary.
- `emax > 1` (e.g. `2`, `5`): the reservoir is unstable and readout noise dominates.

All other settings (frozen input/recurrent, readout seed, training budget) are kept identical.


In [ ]:
EMAX_VALUES = [0.1, 0.5, 1.0, 2.0, 5.0]
MAX_STEPS = 1500

results = {}  # emax -> accuracy

for emax in EMAX_VALUES:
    save_dir = str(MODEL_DIR / f'ctrnn_esn_emax_{emax}')
    if os.path.exists(os.path.join(save_dir, 'model.safetensors')):
        print(f'Loading readout-only ESN with emax={emax} from {save_dir}')
        model = AutoModel.from_pretrained(save_dir)
    else:
        print(f'Training readout-only ESN with emax={emax}')
        torch.manual_seed(42)  # same readout/input initialization for every emax
        cfg = AutoConfig.for_model(
            'ctrnn',
            input_dim=ds.input_dim,
            latent_dim=LATENT_DIM,
            output_dim=ds.output_dim,
            dt=100,
            tau=100,
            nonlinearity_mode='pre_activation',
            freeze_input=True,
            freeze_recurrent=True,
            freeze_output=False,
        )
        model = AutoModel.from_config(cfg)
        with torch.no_grad():
            model.h2h.weight.copy_(critical_init_recurrent(model.h2h.weight, emax=emax, seed=0))

        Trainer(
            model, ds, SupervisedObjective(task_type='classification'),
            TrainingArguments(max_steps=MAX_STEPS, learning_rate=1e-3, log_every=0)
        ).train()

        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)

    acc, _, _ = collect_activity_and_accuracy(model, ds_eval)
    results[emax] = acc
    print(f'emax={emax} -> accuracy={acc:.3f}')

# add the critical model result
results[0.998] = acc_esn
print('All results:', results)

In [ ]:
# Bar plot: accuracy vs emax
labels = [str(k) for k in sorted(results.keys())]
accuracies = [results[float(k)] for k in labels]
colors_bar = ['#d95f02' if float(l) == 0.998 else '#7570b3' for l in labels]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, accuracies, color=colors_bar, edgecolor='k', alpha=0.85)
ax.axhline(0.5, color='gray', linestyle='--', lw=1, label='chance')
ax.set_xlabel('emax (target largest real eigenvalue)', fontsize=10)
ax.set_ylabel('Choice accuracy', fontsize=10)
ax.set_title('ESN performance depends critically on reservoir initialization', fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# annotate bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.annotate(f'{acc:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
fig.savefig(FIG_DIR / 'esn_accuracy_vs_emax.png', dpi=150)
plt.show()


### Interpretation

- **`emax = 0.998`** reaches high accuracy: the reservoir sits near the edge of stability, so it can maintain stimulus evidence across the trial.
- **`emax = 0.1` and `0.5`**: the recurrent dynamics are overdamped. Activity decays too quickly for the readout to recover the integrated evidence, so accuracy drops toward chance.
- **`emax = 1`**: exactly marginal. Performance is typically intermediate because the reservoir is neither strongly damped nor explosively unstable.
- **`emax = 2` and `5`**: the spectral radius exceeds 1, making the linearized reservoir unstable. Hidden states amplify over time and readout noise dominates, again degrading accuracy.

This confirms the central message of the `critical_init` paper for ESN-style training: **reservoir initialization is the dominant factor**, and only a narrow range near criticality yields good task performance when the readout alone is trained.


## 8. Summary

- NeuralRNN's built-in `freeze_*` config flags make ESN-style training straightforward: set `freeze_input=True` and `freeze_recurrent=True`, then train only the readout.
- A CTRNN reservoir initialized with the `critical_init` recipe (`emax ≈ 0.998`) can solve perceptual decision-making with readout-only training.
- Fixed-point analysis under the 0-coherence input reveals the dynamical scaffold used by the readout.
- Comparing reservoirs initialized with `emax = 0.1, 0.5, 1, 2, 5` shows that performance collapses away from the critical initialization, highlighting the importance of the spectral radius for reservoir computing.
